In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_csv("../data/raw/bank-full.csv", sep=";")
num_cols = df.select_dtypes(include="int").columns
cat_cols = df.select_dtypes(include="str").columns
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.info()

This dataset does not have missing values, duplicates and columns are either str or int. Number of rows is 45211 and number of columns is 17. Target name - y, tells us has the client subscribed a term deposit after campaign calls?

In [ ]:
df.describe()

Few things that caught my eye are negative balance i think it's possible but i will research this anyway, outliers especially in columns pdays, previous (also this two can correlate which can harm our model in future), duration looks strange. From data dictionary -1 in pdays means client was not previously contacted)

In [ ]:
for col in df.select_dtypes(include="str").columns:
    print("Column: ", end="")
    print(df[col].value_counts())

In [ ]:
df["y"].value_counts(normalize=True)

First of all target is very imbalanced 0.88 to 0.12 i need to be cautious with this one cause model who just always predict no will have accuracy of 0.88. I will use either precision or recall and instead of ROC-AUC i will use PR-AUC that will be more fair. After that poutcome is interesting column and can tell more to the model however 37k is unknown. Also default column is interesting i think if bank will provide them loan this people with chance will accept it because other banks reject them. Other columns seems to be normal but with imputed category "Unknown"

In [ ]:
for col in num_cols:
    g = sns.histplot(df, x=col)
    g.figure.suptitle(f"{col} distribution")
    plt.show()

Every column beside age, day are highly right skewed.

In [ ]:
sns.histplot(df["balance"], log_scale=True)
plt.show()

Log of the balance look a little better but still exclude negative values. I will use some sort of transfromation on this column rather than default balance because of it skewness. I want to create bins and understand what is that peak is this exactly zero or other number.

In [ ]:
df[df["balance"] == 0].shape

In [ ]:
balance_bins = [-9000, 0, 1, 5000, 20000, np.inf]
balance_labels = ["negative", "zero", "low", "medium", "high"]
balance_df = df.copy()
balance_df["balance_size"] = pd.cut(
    balance_df["balance"], bins=balance_bins, labels=balance_labels, right=False
)
balance_df["balance_size"].value_counts()

Actually majority of balance are not zero or negative but is just a low amount of money. Now i want to see event rate for each numeric column in order to understand shape of the relationship between numeric columns and target. Thus understand what to do with each column.

In [ ]:
y_num = (df["y"] == "yes").astype(int)

for col in num_cols:
    bins = pd.qcut(df[col], 10, duplicates="drop")
    rates = y_num.groupby(bins, observed=True).mean()
    print(f"{col} : {rates}")

Many interesting things i found. 1)Age relationship is not linear and look like U-shaped rel cause we have young people with high rate then middle-aged people with lower and then seniors with higher rate. 2)Balance here we see strong linear relationship higher balance = higher rate. 3) Day is not important and i will drop it probably. 4) The most interesting Duration what we see here very strong linear relationship longer duration = higher rate but there are two problems. To start with we are trying to predict who we need to call but information about duration of the call we will get only after the call. In addition, low duration = instant no but higher durations = customer is interested = yes, but actually it is reversed interested customer = yes and not interested = no. 5) Campaign or number of call during it. what we see here more call = less yes , it is because people can become annoyed and if people are interested they will accept offer in first calls. 6) pdays and previous are highly skewed and is hard to tell something.


I looked at the data dictionary and my thoughts about duration was right and author suggest to drop this column

```
last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
```